In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
sys.path.insert(0, str(PLL_DIR.parents[1]))  # repo root
from sw.lib.pll_util import bin_phase_noise

# Debug of the divider dependence of the in-band phase noise (tests/pll_spectrum_div.py). Every run: 1 GHz VCO,
# codes c2/11/4 (|Kvco| ~1.15 GHz/V), reference 7.8125 MHz from the 33600A, pad -> DC blocker -> FSV.
# (step, condition, CSV); vddio is set by hand on the N6705B, so it is only known from here.
RUNS = [
    ("baseline", "7 Oct, filter not recorded", "pll_spectrum_div_S5_20261007_112110.csv"),
    ("baseline", "13 Oct, filter 7/0/0/7", "pll_spectrum_div_S5_20261013_142509.csv"),
    ("1 repeat", "/8 /128 /8 /128 /8", "pll_spectrum_div_S5_20261014_145301_repeat.csv"),
    ("4a ref amplitude", "ref 1.8 V", "pll_spectrum_div_S5_20261014_154110_vpp1p8.csv"),
    ("4a ref amplitude", "ref 1.2 V", "pll_spectrum_div_S5_20261014_154205_vpp1p2.csv"),
    ("4a ref amplitude", "ref 0.9 V", "pll_spectrum_div_S5_20261014_154259_vpp0p9.csv"),
    ("4a ref amplitude", "ref 1.8 V (again)", "pll_spectrum_div_S5_20261014_154316_vpp1p8.csv"),
    ("4b PLL VDD", "VDD 0.75 V", "pll_spectrum_div_S5_20261014_160345_vdd0p75.csv"),
    ("4b PLL VDD", "VDD 0.70 V", "pll_spectrum_div_S5_20261014_160446_vdd0p70.csv"),
    ("4b PLL VDD", "VDD 0.65 V", "pll_spectrum_div_S5_20261014_160547_vdd0p65.csv"),
    ("4b PLL VDD", "VDD 0.75 V (again)", "pll_spectrum_div_S5_20261014_160647_vdd0p75.csv"),
    ("4b VDDIO", "VDDIO 1.8 V", "pll_spectrum_div_S5_20261014_160808_vddio1p80.csv"),
    ("4b VDDIO", "VDDIO 1.6 V", "pll_spectrum_div_S5_20261014_160937_vddio1p60.csv"),
    ("4b VDDIO", "VDDIO 1.5 V", "pll_spectrum_div_S5_20261014_161052_vddio1p50.csv"),
    ("4b VDDIO", "VDDIO 1.8 V (again)", "pll_spectrum_div_S5_20261014_161215_vddio1p80.csv"),
]
TONES = {"4818 Hz": (4700, 4950), "794 kHz": (790e3, 800e3)}  # in-band / out-of-band test tones: offset window


def tone_dbc(spurs, lo, hi):
    """Strongest spur [dBc on the pad] in the `spurs` column between offsets lo and hi [Hz]; NaN if none."""
    found = [float(lv) for f, lv in re.findall(r"([\d.]+) Hz: (-?[\d.]+) dBc", str(spurs)) if lo <= float(f) <= hi]
    return max(found) if found else np.nan


frames = []
for i, (step, cond, name) in enumerate(RUNS):
    d = pd.read_csv(PLL_DIR / name)
    frames.append(d.assign(run=i, step=step, condition=cond, csv=name, row_in_run=np.arange(len(d))))
res = pd.concat(frames, ignore_index=True)
res["n_vco"] = 20 * np.log10(res["total_div"])  # pad dBc + n_vco = referred to the VCO
for tone, (lo, hi) in TONES.items():
    res[f"{tone} pad"] = [tone_dbc(s, lo, hi) for s in res["spurs"]]
    res[f"{tone} vco"] = res[f"{tone} pad"] + res["n_vco"]
res["meas"] = res["divider"] + np.where(res.groupby(["run", "divider"]).cumcount() > 0,
                                        " #" + (res.groupby(["run", "divider"]).cumcount() + 1).astype(str), "")

pd.DataFrame({
    "step": res["step"], "condition": res["condition"], "divider": res["meas"], "f_pad [MHz]": res["f_out"] / 1e6,
    "locked": res["locked"], "carrier [dBm]": res["carrier_dbm"],
    "L_vco 1k": res["L_vco_1000"], "L_vco 10k": res["L_vco_10000"], "L_vco 100k": res["L_vco_100000"],
    "L_vco 1M": res["L_vco_1000000"], "4818 Hz at VCO [dBc]": res["4818 Hz vco"],
    "794 kHz at VCO [dBc]": res["794 kHz vco"], "noise jitter [ps]": res["time_jitter_noise_ps"],
    "spur jitter [ps]": res["time_jitter_spur_ps"], "ref [V]": res.get("ref_vpp"), "PLL VDD [V]": res["pll_vdd"],
    "PLL I [mA]": res["pll_i"] * 1e3, "status": res["status"],
}).round(2)

In [ ]:
INK, GRID = "#52514e", "#e4e3df"
FREQ_AXIS = dict(type="log", tickvals=[1, 10, 100, 1e3], ticktext=["1 MHz", "10 MHz", "100 MHz", "1 GHz"],
                 minor=dict(showgrid=True, gridcolor="#f1f0ed"))
OFFSET_COLORS = {"1000": "#2a78d6", "100000": "#eb6834", "1000000": "#1baf7a"}
OFFSET_NAMES = {"1000": "1 kHz", "100000": "100 kHz", "1000000": "1 MHz"}
TONE_COLORS = {"4818 Hz": "#e34948", "794 kHz": "#4a3aa7"}

# The problem (13 Oct sweep, every divider): referred to the VCO the noise should be the same for every divider.
# Left: inside the loop bandwidth (1k, 100k) it rises ~15 dB from a 7.8 MHz to a 125 MHz pad, at 1 MHz it is flat.
# Right: test tones referred to the VCO. The 794 kHz tone (outside the loop bandwidth) is flat: analyzer, DC blocker
# and the +20 log10 N referral are right. The 4818 Hz tone (inside the loop bandwidth) grows with f_pad like the noise.
base = res[(res["condition"] == "13 Oct, filter 7/0/0/7") & (res["status"] == "ok")].sort_values("f_out")
symbols = np.where(base["pll_clk_o_en"] == 1, "circle", "diamond")
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                    subplot_titles=("phase noise referred to the VCO", "test tones referred to the VCO"))
for off, color in OFFSET_COLORS.items():
    fig.add_scatter(x=base["f_out"] / 1e6, y=base[f"L_vco_{off}"], mode="markers+lines",
                    name=f"L at {OFFSET_NAMES[off]}",
                    line=dict(color=color, width=1, dash="dot"),
                    marker=dict(size=10, color=color, symbol=symbols, line=dict(color="white", width=1)),
                    customdata=base[["divider"]], row=1, col=1,
                    hovertemplate="%{customdata[0]}: pad %{x:.4g} MHz, %{y:.1f} dBc/Hz<extra></extra>")
for tone, color in TONE_COLORS.items():
    d = base[base[f"{tone} vco"].notna()]
    fig.add_scatter(x=d["f_out"] / 1e6, y=d[f"{tone} vco"], mode="markers+lines", name=f"{tone} tone",
                    line=dict(color=color, width=1, dash="dot"),
                    marker=dict(size=10, color=color, symbol=np.where(d["pll_clk_o_en"] == 1, "circle", "diamond"),
                                line=dict(color="white", width=1)),
                    customdata=d[["divider", f"{tone} pad"]], row=1, col=2,
                    hovertemplate="%{customdata[0]}: pad %{x:.4g} MHz, %{y:.1f} dBc at the VCO "
                                  "(%{customdata[1]:.1f} dBc on the pad)<extra></extra>")
fig.update_layout(title="Baseline 13 Oct: in-band noise and the 4818 Hz tone grow with f_pad, 1 MHz and the 794 kHz "
                        "tone do not<br><sup>circle: divider inside the PLL, diamond: external divider</sup>",
                  template="plotly_white", height=480, legend=dict(orientation="h", y=-0.22))
fig.update_xaxes(title="pad frequency", gridcolor=GRID, **FREQ_AXIS)
fig.update_yaxes(title="L referred to the VCO [dBc/Hz]", gridcolor=GRID, row=1, col=1)
fig.update_yaxes(title="tone referred to the VCO [dBc]", gridcolor=GRID, row=1, col=2)
fig.show()

In [ ]:
STEPS = ["1 repeat", "4a ref amplitude", "4b PLL VDD", "4b VDDIO"]
QUANTITIES = [("L_vco_1000", "L at 1 kHz [dBc/Hz]"), ("L_vco_100000", "L at 100 kHz [dBc/Hz]"),
              ("4818 Hz vco", "4818 Hz tone [dBc]")]
DIV_COLORS = {"int /8": "#2a78d6", "int /128": "#eb6834"}

# Debug steps, /8 (125 MHz pad) against /128 (7.8 MHz pad), everything referred to the VCO. One column per step,
# x = the condition in measurement order (step 1: the measurement number). A path that carries the pad into the loop
# would show as the /8 - /128 gap changing with the condition. Not locked = no marker (ref 0.9 V).
steps = res[res["step"].isin(STEPS) & (res["status"] == "ok")].copy()
steps["x"] = np.where(steps["step"] == "1 repeat", "meas " + (steps["row_in_run"] + 1).astype(str), steps["condition"])
fig = make_subplots(rows=len(QUANTITIES), cols=len(STEPS), shared_yaxes="rows", horizontal_spacing=0.03,
                    vertical_spacing=0.08, subplot_titles=STEPS)
for c, step in enumerate(STEPS, 1):
    s = steps[steps["step"] == step]
    order = list(dict.fromkeys(s.sort_values(["run", "row_in_run"])["x"]))
    for r, (col, ylabel) in enumerate(QUANTITIES, 1):
        for div, color in DIV_COLORS.items():
            d = s[s["divider"] == div]
            label = f"{div} (pad {d['f_out'].iloc[0] / 1e6:.4g} MHz)" if len(d) else div
            fig.add_scatter(x=d["x"], y=d[col], mode="markers+lines", name=label,
                            legendgroup=div, showlegend=(r == 1 and c == 1),
                            line=dict(color=color, width=1, dash="dot"),
                            marker=dict(size=10, color=color, line=dict(color="white", width=1)),
                            customdata=d[["condition", "csv"]], row=r, col=c,
                            hovertemplate=f"{div}, %{{customdata[0]}}<br>%{{y:.1f}}<br>%{{customdata[1]}}"
                                          "<extra></extra>")
        fig.update_xaxes(categoryorder="array", categoryarray=order, row=r, col=c)
        if c == 1:
            fig.update_yaxes(title=ylabel, row=r, col=c)
fig.update_layout(title="Debug steps: /8 vs /128 referred to the VCO per condition (the gap stays 10-16 dB in every "
                        "step)", template="plotly_white", height=850, legend=dict(orientation="h", y=-0.12))
fig.update_xaxes(gridcolor=GRID, tickangle=-30)
fig.update_yaxes(gridcolor=GRID)
fig.show()

# /8 - /128 gap per condition (dB, positive = /8 worse)
gap = (steps.groupby(["step", "condition", "run", "divider"])[[q for q, _ in QUANTITIES]].mean().unstack("divider"))
gap = pd.DataFrame({ylabel: gap[(q, "int /8")] - gap[(q, "int /128")] for q, ylabel in QUANTITIES})
gap.sort_index(level="run").droplevel("run").round(1)

In [ ]:
CURVE_RUNS = ["ref 1.8 V", "ref 1.2 V", "VDD 0.65 V", "VDDIO 1.5 V"]  # conditions with one /8 and one /128 each
F_start, F_stop, N_BINS = 1e3, 10e6, 300
OFFSET_AXIS = dict(type="log", tickvals=[1e3, 1e4, 1e5, 1e6, 1e7], ticktext=["1 kHz", "10 kHz", "100 kHz", "1 MHz",
                                                                            "10 MHz"],
                   range=[np.log10(F_start), np.log10(F_stop)], minor=dict(showgrid=True, gridcolor="#f1f0ed"))
DASHES = ["solid", "dot", "dash", "dashdot"]

# Full phase-noise curves (L_trace) referred to the VCO, /8 (blue) and /128 (orange), for a few conditions (line
# style). Every /8 curve lies above every /128 curve inside the loop bandwidth and they meet around 1 MHz; the
# conditions hardly move the curves. Binned to N_BINS points (linear power average, display only).
fig = go.Figure()
for k, cond in enumerate(CURVE_RUNS):
    name = next(n for _, c, n in RUNS if c == cond)
    tr = pd.read_csv(PLL_DIR / name.replace(".csv", "_traces.csv"))
    for div, color in DIV_COLORS.items():
        lt = tr[(tr["divider"] == div) & (tr["kind"] == "L_trace")]
        lt = lt[(lt["freq"] >= F_start) & (lt["freq"] <= F_stop)].sort_values("freq")
        if lt.empty:
            continue
        freq, level = bin_phase_noise(lt["freq"], lt["level_dbm"], np.geomspace(F_start, F_stop, N_BINS + 1))
        n_vco = 20 * np.log10(lt["total_div"].iloc[0])
        fig.add_scatter(x=freq, y=level + n_vco, mode="lines", name=f"{div}, {cond}",
                        line=dict(color=color, width=1.5, dash=DASHES[k % len(DASHES)]),
                        hovertemplate=f"{div}, {cond}<br>%{{x:.3s}}Hz: %{{y:.1f}} dBc/Hz at the VCO<extra></extra>")
fig.update_layout(title="Phase noise referred to the VCO: /8 (125 MHz pad) vs /128 (7.8 MHz pad) per condition",
                  template="plotly_white", height=550, legend=dict(orientation="h", y=-0.2, font=dict(size=10)),
                  hoverlabel=dict(namelength=-1))
fig.update_xaxes(title="offset from the carrier", gridcolor=GRID, **OFFSET_AXIS)
fig.update_yaxes(title="L referred to the VCO [dBc/Hz]", gridcolor=GRID)
fig.show()

In [ ]:
# Status of the debug (14 Oct 2026); update with every step.
pd.set_option("display.max_colwidth", None)
pd.DataFrame([
    ("baseline", "Is the in-band noise the same for every divider (referred to the VCO)?",
     "No: 1k-100 kHz ~15 dB worse at a 125 MHz pad than at 7.8 MHz; 1 MHz flat; int /8 = ext /2 x /4 (same pad f)",
     "depends on f_pad, not on the divider path"),
    ("2 chain", "Do analyzer, DC blocker, referral distort it?",
     "794 kHz tone (outside the loop) is flat within +-2.5 dB at the VCO over all dividers; carrier level follows the "
     "DC blocker (seen on the scope) but not the noise", "ruled out"),
    ("1 repeat", "Real and repeatable, or drift / sweep order?",
     "/8 - /128 ~11 dB at 1k and 100k on every repeat; /128 within 0.2 dB, /8 drifts ~2 dB worse over 1.5 min",
     "real"),
    ("4a ref amplitude", "Coupling at the reference edge (error = V_coupled / slope)?",
     "ref 1.8 -> 1.2 V (slope -3.5 dB): /8 moves ~1 dB, inside the scatter; 0.9 V does not lock",
     "unlikely (only 3.5 dB of slope range)"),
    ("4b PLL VDD", "Does the PLL supply level change the gap?",
     "VDD 0.75/0.70/0.65 V: /8 ~3 dB better below 0.75 V, gap stays 10-16 dB, not monotonic",
     "no clear effect (DC level only, not the supply impedance)"),
    ("4b VDDIO", "Does a smaller pad swing / current help?",
     "VDDIO 1.8/1.6/1.5 V: carrier -2.4 dB at 1.5 V, noise and gap unchanged within the scatter",
     "inconclusive (only ~2 dB less pad current)"),
    ("open", "Separate finding: strong tones near 2 MHz", "1.95 / 2.12 / 2.39 MHz at ~-17..-24 dBc on the /8 pad, "
     "~-4 dBc at the VCO, flat over the dividers, frequency moves between days (716 kHz on 7 Oct)",
     "loop peaking / low phase margin with C1 = C2 = 0? dominates full-band jitter"),
], columns=["step", "question", "result", "conclusion"])